# Advanced FileSet Features

This notebook demonstrates advanced FileSet capabilities:
- **Metadata filtering** — restrict which documents are used as seeds
- **Query-based seeds** — use semantic queries instead of chunking
- **RAG context generation** — retrieve supporting context with temporal constraints
- **RAG labeling** — resolve questions by searching the FileSet
- **Full combined pipeline** — context + labeling in one run

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [1]:
%pip install lightningrod-ai python-dotenv pandas -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [ ]:
fileset_id = "PASTE_YOUR_FILESET_ID_HERE"

In [7]:
import pandas as pd
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    FileSetQuerySeedGenerator,
    FileSetContextGenerator,
    FileSetRAGLabeler,
    QuestionGenerator,
    BinaryAnswerType,
    TemporalConstraint,
)

answer_type = BinaryAnswerType()

## Metadata Filtering

Use `metadata_filters` on the seed generator to restrict which files become seeds. Here we generate questions only from **APEX** documents.

In [8]:
pipeline_filtered = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='APEX'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the specific financial metrics and business events in these quarterly reports.",
        questions_per_seed=2,
    ),
)

dataset_filtered = lr.transforms.run(
    pipeline_filtered,
    max_questions=6,  # Increase to ~10000 for a real run
    name="FileSet - APEX Only (Metadata Filter)",
)
print(f"Dataset: {dataset_filtered.id}")
print(f"Rows: {dataset_filtered.num_rows}")

Dataset: 23650fa3-341a-4725-89d1-e206aa6f572f
Rows: 6


In [9]:
samples_filtered = dataset_filtered.download()
for i, s in enumerate(samples_filtered[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Seed (first 120 chars): {s.seed.seed_text[:120]}...")
    print(f"Question: {s.question.question_text}")
    print()

--- Sample 1 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q1 2024
Period ending March 31, 2024

Financial Highlights:
- Revenu...
Question: Did APEX Technologies Inc. report a year-over-year revenue increase of more than 10% in Q1 2024?

--- Sample 2 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q1 2025
Period ending March 31, 2025

Financial Highlights:
- Revenu...
Question: Has the number of AXCompute 3.0 enterprise clients reached at least 90?

--- Sample 3 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q1 2024
Period ending March 31, 2024

Financial Highlights:
- Revenu...
Question: Has the AXCompute 3.0 platform secured commitments from more than 10 Fortune 500 clients according to the report?



## Query-Based Seeds

`FileSetQuerySeedGenerator` runs semantic queries against the FileSet instead of chunking all documents. This is useful when you want seeds focused on specific topics. Here we query **VGI** documents only.

In [10]:
pipeline_query = QuestionPipeline(
    seed_generator=FileSetQuerySeedGenerator(
        file_set_id=fileset_id,
        prompts=[
            "What is the current status and growth trajectory of the Robotics-as-a-Service (RaaS) segment?",
            "What restructuring actions has the company taken and what are the expected cost savings?",
            "What are the company's revenue guidance figures for the next quarter and full year?",
        ],
        metadata_filters=["ticker='VGI'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the specific facts in the retrieved content.",
        questions_per_seed=2,
    ),
)

dataset_query = lr.transforms.run(
    pipeline_query,
    max_questions=6,  # Increase to ~10000 for a real run
    name="FileSet - Query Seeds (VGI)",
)
print(f"Dataset: {dataset_query.id}")
print(f"Rows: {dataset_query.num_rows}")

Dataset: bd9855ae-c395-4d1e-9b8d-918c9268273c
Rows: 6


In [11]:
samples_query = dataset_query.download()
rows = dataset_query.flattened()
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "is_valid"]
df[[c for c in cols if c in df.columns]]

,question_text
0,Has Vanguard Industries Inc. completed two out...
1,Does Vanguard Industries Inc. project its RaaS...
2,Did Vanguard Industries Inc. report that its R...
3,Has Vanguard Industries Inc. raised its full-y...
4,Is the maximum anticipated revenue for Vanguar...
5,Did the company achieve $38 million in cost sa...


## RAG Context Generation

`FileSetContextGenerator` retrieves supporting context from the FileSet for each generated question.

- **`metadata_filter_keys=["ticker"]`** — only retrieve context from the same company
- **`temporal_constraint=BEFORE`** — only retrieve context from documents dated before the seed, preventing lookahead bias

In [12]:
pipeline_context = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the financial performance and business events in these investor reports.",
        questions_per_seed=1,
    ),
    context_generators=[
        FileSetContextGenerator(
            file_set_id=fileset_id,
            metadata_filter_keys=["ticker"],
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
)

dataset_context = lr.transforms.run(
    pipeline_context,
    max_questions=5,  # Increase to ~10000 for a real run
    name="FileSet - Context Generation (BEFORE)",
)
print(f"Dataset: {dataset_context.id}")
print(f"Rows: {dataset_context.num_rows}")

/usr/local/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Dataset: f69b1a9d-98ae-4e6a-bd01-6668ebe0a61e
Rows: 5


In [13]:
samples_context = dataset_context.download()
for i, s in enumerate(samples_context[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Question: {s.question.question_text}")
    if s.context:
        for j, ctx in enumerate(s.context):
            rendered = getattr(ctx, 'rendered_context', str(ctx))
            print(f"  Context {j+1} (first 200 chars): {str(rendered)[:200]}...")
    else:
        print("  Context: None")
    print()

--- Sample 1 ---
Question: Does APEX Technologies Inc. plan to achieve general availability for the AXCompute 4.0 preview program in the third quarter of 2025?
  Context 1 (first 200 chars): ---
FILESET CONTEXT
[1] (source: Unknown)
APEX Technologies Inc. — Quarterly Investor Report, Q1 2025
Period ending March 31, 2025

Financial Highlights:
- Revenue: $2.63 billion (up 23% YoY)
- Earnin...

--- Sample 2 ---
Question: Does APEX Technologies Inc. expect to complete the acquisition of CyberShield Corp during the first quarter of 2025?
  Context 1 (first 200 chars): ---
FILESET CONTEXT
[1] (source: Unknown)
APEX Technologies Inc. — Quarterly Investor Report, Q4 2024
Period ending December 31, 2024

Financial Highlights:
- Revenue: $2.58 billion (up 21% YoY)
- Ear...

--- Sample 3 ---
Question: Did APEX Technologies Inc. report a year-over-year revenue increase of 12% in the first quarter of 2024?
  Context 1 (first 200 chars): ---
FILESET CONTEXT
[1] (source: Unknown)
APEX Technologies I

## RAG Labeling

`FileSetRAGLabeler` resolves questions by searching the FileSet for answers.

- **`temporal_constraint=AFTER`** — only search documents dated after the seed, so forward-looking questions are resolved by later reports
- **`confidence_threshold=0.7`** — only label questions where the labeler is at least 70% confident

In [14]:
pipeline_labeler = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='VGI'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements, guidance, and planned initiatives "
            "mentioned in these quarterly reports. Focus on questions whose answers would be found in "
            "subsequent quarterly reports."
        ),
        questions_per_seed=2,
    ),
    labeler=FileSetRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_labeler = lr.transforms.run(
    pipeline_labeler,
    max_questions=6,  # Increase to ~10000 for a real run
    name="FileSet - RAG Labeler (VGI, AFTER)",
)
print(f"Dataset: {dataset_labeler.id}")
print(f"Rows: {dataset_labeler.num_rows}")

Dataset: 61455e2c-a5e8-4bc0-bd02-c6c513106ce5
Rows: 6


In [15]:
samples_labeler = dataset_labeler.download()
rows = dataset_labeler.flattened()
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "reasoning"]
df[[c for c in cols if c in df.columns]]

,question_text,label,label_confidence,reasoning
0,Will Vanguard Industries Inc. reach a decision...,1.0,0.95,"The RAG answer explicitly states, ""Yes, Vangua..."
1,Will the Robotics-as-a-Service (RaaS) segment ...,NaN,0.95,The RAG Answer explicitly states there is no i...
2,Will Vanguard Industries Inc.'s revenue for Q2...,NaN,1.00,The RAG answer explicitly states that there is...
3,Did Vanguard Industries Inc. achieve a quarter...,1.0,1.00,The RAG answer explicitly states that Vanguard...
4,Will Vanguard Industries Inc. complete the clo...,NaN,0.80,The RAG answer confirms that restructuring eff...
5,Will Vanguard Industries Inc. report Q2 2024 r...,1.0,1.00,The RAG answer explicitly states that Vanguard...


## Full Pipeline — Context + Labeling

Combine context generation and labeling in a single pipeline:
- **Context** (`BEFORE`) — retrieve earlier reports as supporting context
- **Labeler** (`AFTER`) — resolve forward-looking questions using later reports

In [16]:
pipeline_full = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements and guidance in these investor reports. "
            "Focus on questions that can be verified by looking at later quarterly reports for the same company."
        ),
        questions_per_seed=1,
    ),
    context_generators=[
        FileSetContextGenerator(
            file_set_id=fileset_id,
            metadata_filter_keys=["ticker"],
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
    labeler=FileSetRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_full = lr.transforms.run(
    pipeline_full,
    max_questions=8,  # Increase to ~10000 for a real run
    name="FileSet - Full Pipeline (Context + Labeler)",
)
print(f"Dataset: {dataset_full.id}")
print(f"Rows: {dataset_full.num_rows}")

/usr/local/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Dataset: f847bcc6-a34e-416b-86e4-d3a11ecd9a76
Rows: 8


In [17]:
samples_full = dataset_full.download()
for i, s in enumerate(samples_full[:4]):
    print(f"--- Sample {i+1} ---")
    print(f"Question: {s.question.question_text}")
    if s.context:
        for j, ctx in enumerate(s.context):
            rendered = getattr(ctx, 'rendered_context', str(ctx))
            print(f"  Context {j+1}: {str(rendered)}")
    else:
        print("  Context: None")
    if s.label:
        print(f"  Label: {s.label.label} (confidence: {s.label.label_confidence})")
        print(f"  Reasoning: {s.label.reasoning}")
    else:
        print("  Label: None")
    print()

--- Sample 1 ---
Question: Will APEX Technologies Inc. close the acquisition of CyberShield Corp in the first quarter of 2025?
  Context 1: ---
FILESET CONTEXT
[1] (source: Unknown)
APEX Technologies Inc. — Quarterly Investor Report, Q4 2024
Period ending December 31, 2024

Financial Highlights:
- Revenue: $2.58 billion (up 21% YoY)
- Earnings per share (EPS): $2.24
- Operating margin: 20.2%
- Free cash flow: $420 million
- Full-year 2024 revenue: $9.41 billion

Business Update:
Record quarter for APEX. The Northern Virginia data center came online in November, adding 35% more compute capacity. Total AXCompute 3.0 enterprise clients reached 74. The company completed due diligence on CyberShield Corp, a cybersecurity firm, and expects to close the $980 million acquisition in Q1 2025. International revenue now at 37% of total.

Guidance:
FY2025 revenue guidance: $10.5B-$11.0B (12-17% growth). Q1 2025 revenue expected at $2.55B-$2.65B, with margin temporarily impacted by CyberShield integ

In [18]:
rows = dataset_full.flattened()
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "reasoning", "is_valid"]
df[[c for c in cols if c in df.columns]]

,question_text,label,label_confidence,reasoning
0,Will APEX Technologies Inc. close the acquisit...,1.0,1.0,"The RAG answer explicitly states, 'Yes, APEX T..."
1,Will APEX Technologies Inc. report Q2 2025 rev...,NaN,1.0,The RAG Answer explicitly states that there is...
2,Will APEX Technologies Inc. report Q3 2024 rev...,1.0,1.0,The RAG answer explicitly states that APEX Tec...
3,Did APEX Technologies Inc. report revenue of a...,1.0,1.0,"The RAG answer explicitly states 'Yes, APEX Te..."
4,Will APEX Technologies Inc. report a revenue o...,1.0,1.0,The RAG answer explicitly states that APEX Tec...
5,Will Vanguard Industries Inc. report a total r...,1.0,1.0,The RAG answer explicitly states that Vanguard...
6,Will Vanguard Industries Inc. report total rev...,NaN,1.0,The RAG answer explicitly states there is no i...
7,Will Vanguard Industries Inc. report total rev...,1.0,1.0,"The RAG answer explicitly states 'Yes, Vanguar..."
